# Budgerigar R0：连续波形自编码器
不使用 codec token、token tape、硬指针或输入直通；真实训练与试听在 Colab T4 进行。

In [ ]:
#@title 1. 可恢复地更新项目
REPO_DIR='/content/Budgerigar'
from pathlib import Path
import subprocess,sys,importlib,shutil,datetime
repo=Path(REPO_DIR)
if not (repo/'.git').is_dir():subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
else:
 pull=subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],text=True,capture_output=True);print(pull.stdout,pull.stderr)
 if pull.returncode:
  backup=repo.with_name(f'Budgerigar_backup_{datetime.datetime.now():%H%M%S}');shutil.move(str(repo),str(backup));subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[train]'],check=True);sys.path.insert(0,REPO_DIR)
for name in [k for k in list(sys.modules) if k=='budgerigar' or k.startswith('budgerigar.')]:del sys.modules[name]
importlib.invalidate_caches();commit=subprocess.run(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip();print('commit:',commit)

In [ ]:
#@title 2. Drive 与结构检查
from google.colab import drive
drive.mount('/content/drive');WORK_ROOT=Path('/content/drive/MyDrive/Budgerigar');SOURCE_MANIFEST=WORK_ROOT/'manifests'/'cmu_arctic.jsonl'
import torch,json
if not torch.cuda.is_available():raise RuntimeError('请选择 GPU runtime')
from budgerigar.continuous_audio_data import WaveformChunkDataset
from budgerigar.continuous_autoencoder import ContinuousAutoencoderConfig,create_continuous_autoencoder
model_config=ContinuousAutoencoderConfig();preview=WaveformChunkDataset(SOURCE_MANIFEST,'train',max_records=2,random_crop=False);waveform=preview[0][0].unsqueeze(0);model=create_continuous_autoencoder(model_config)
with torch.no_grad():reconstruction,latent=model(waveform)
parameters=sum(p.numel() for p in model.parameters());print(torch.cuda.get_device_name(0),parameters,waveform.shape,latent.shape,reconstruction.shape)
assert reconstruction.shape==waveform.shape and latent.shape[1]==75

In [ ]:
#@title 3. T4 R0 smoke training
MAX_STEPS=400 #@param {type:'integer'}
BATCH_SIZE=4 #@param {type:'integer'}
from budgerigar.train_continuous_autoencoder import AutoencoderTrainingConfig,train_continuous_autoencoder
RUN_DIR=WORK_ROOT/'checkpoints'/'continuous_autoencoder_r0';training=AutoencoderTrainingConfig(batch_size=BATCH_SIZE,max_steps=MAX_STEPS)
report=train_continuous_autoencoder(SOURCE_MANIFEST,RUN_DIR,training,model_config);print(json.dumps(report,ensure_ascii=False,indent=2))

In [ ]:
#@title 4. 试听与门槛
import torchaudio
from IPython.display import Audio,display
sample=torch.load(RUN_DIR/'validation_example.pt',map_location='cpu',weights_only=True);INPUT_WAV=RUN_DIR/'validation_input.wav';RECON_WAV=RUN_DIR/'validation_reconstruction.wav'
torchaudio.save(str(INPUT_WAV),sample['input'].unsqueeze(0),sample['sample_rate']);torchaudio.save(str(RECON_WAV),sample['reconstruction'].unsqueeze(0),sample['sample_rate'])
display(Audio(str(INPUT_WAV)));display(Audio(str(RECON_WAV)));best=min(report['history'],key=lambda x:x['validation_spectral_loss']);r0_pass=best['validation_si_sdr_db']>5
print(json.dumps(best,ensure_ascii=False,indent=2),'r0_pass=',r0_pass)
from budgerigar.experiment import write_run_metadata
write_run_metadata(RUN_DIR/'run_metadata.json',SOURCE_MANIFEST,{'architecture':'continuous_waveform_autoencoder','parameters':parameters,'best_validation_si_sdr_db':best['validation_si_sdr_db'],'r0_pass':r0_pass},repository=REPO_DIR)